<a href="https://colab.research.google.com/github/suchismitab0511/accident_pred/blob/main/accident_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df=pd.read_csv('/content/sc_accidents.csv')
df.head()

/tmp/ipython-input-1-922903210.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('/content/sc_accidents.csv')


,serial_number,severity,start_lat,start_lng,end_lat,end_lng,temperature_f,visibility_mi
0,1,2,34.574226,-79.54068,NaN,NaN,44.6,2.5
1,2,3,34.833031,-82.296837,NaN,NaN,73.9,10.0
2,3,2,34.318562,-82.663651,NaN,NaN,64.9,0.8
3,4,2,34.202515,-82.134941,NaN,NaN,63.0,4.0
4,5,3,34.293327,-81.545921,NaN,NaN,62.6,10.0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df.isnull().sum()


,0
serial_number,0
severity,0
start_lat,0
start_lng,0
end_lat,235965
end_lng,235965
temperature_f,4915
visibility_mi,5443


In [ ]:
df=df.drop(columns=['serial_number','end_lat','end_lng'])
df['temperature_f'].fillna(df['temperature_f'].median(), inplace=True)
df['visibility_mi'].fillna(df['visibility_mi'].median(), inplace=True)


/tmp/ipython-input-4-4023086665.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['temperature_f'].fillna(df['temperature_f'].median(), inplace=True)
/tmp/ipython-input-4-4023086665.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(val

In [ ]:
X = df.drop(columns=['severity'])
y = df['severity']

In [ ]:
print(y.value_counts())


severity
2    232170
3     41673
1      6112
4      1317
Name: count, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


ValueError: could not convert string to float: '-'

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)


y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)




In [ ]:
import matplotlib.pyplot as plt


n = 50
plt.figure(figsize=(16, 6))

plt.plot(y_test.values[:n], label='Actual Severity', marker='o')
plt.plot(y_pred[:n], label='Predicted Severity', marker='x')

plt.title("Actual vs Predicted Accident Severity (First 50 Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Severity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# XGBoost


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt


df = pd.read_csv("sc_accidents.csv")
df = df.drop(columns=['serial_number', 'end_lat', 'end_lng'])
df['temperature_f'] = df['temperature_f'].fillna(df['temperature_f'].median())
df['visibility_mi'] = df['visibility_mi'].fillna(df['visibility_mi'].median())


X = df.drop(columns=['severity'])
y = df['severity'] - 1


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)
xgb.fit(X_train_scaled, y_train)


y_pred = xgb.predict(X_test_scaled)
y_test = y_test + 1
y_pred = y_pred + 1


print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [18:07:46] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Accuracy: 0.8837

Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00      1219
           2       0.89      0.99      0.94     66233
           3       0.72      0.26      0.38      8455
           4       0.80      0.01      0.01       605

    accuracy                           0.88     76512
   macro avg       0.60      0.31      0.33     76512
weighted avg       0.86      0.88      0.85     76512



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE


df = pd.read_csv("sc_accidents.csv")
df = df.drop(columns=['serial_number', 'end_lat', 'end_lng'])
df['temperature_f'] = df['temperature_f'].fillna(df['temperature_f'].median())
df['visibility_mi'] = df['visibility_mi'].fillna(df['visibility_mi'].median())


X = df.drop(columns=['severity'])
y = df['severity'] - 1


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=4,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)
xgb.fit(X_train_resampled, y_train_resampled)


y_pred = xgb.predict(X_test_scaled)
y_test = y_test + 1
y_pred = y_pred + 1


print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))





/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [18:09:30] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Accuracy: 0.5478

Classification Report:
              precision    recall  f1-score   support

           1       0.05      0.61      0.09      1219
           2       0.97      0.52      0.67     66233
           3       0.38      0.80      0.52      8455
           4       0.02      0.24      0.03       605

    accuracy                           0.55     76512
   macro avg       0.35      0.54      0.33     76512
weighted avg       0.88      0.55      0.64     76512



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt


df = pd.read_csv("sc_accidents.csv")
df = df.drop(columns=['serial_number', 'end_lat', 'end_lng'])

df['temperature_f'] = df['temperature_f'].fillna(df['temperature_f'].median())
df['visibility_mi'] = df['visibility_mi'].fillna(df['visibility_mi'].median())


df['severity_binary'] = df['severity'].apply(lambda x: 0 if x in [1, 2] else 1)


X = df.drop(columns=['severity', 'severity_binary'])
y = df['severity_binary']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


log_reg = LogisticRegression(class_weight='balanced', max_iter=1000)
log_reg.fit(X_train_scaled, y_train)


y_pred = log_reg.predict(X_test_scaled)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))





Accuracy: 0.5443

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.54      0.68     67452
           1       0.14      0.55      0.22      9060

    accuracy                           0.54     76512
   macro avg       0.52      0.55      0.45     76512
weighted avg       0.81      0.54      0.62     76512



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


df = pd.read_csv("sc_accidents.csv")
df = df.drop(columns=['serial_number', 'end_lat', 'end_lng', 'start_lat', 'start_lng'])

df['temperature_f'] = df['temperature_f'].fillna(df['temperature_f'].median())
df['visibility_mi'] = df['visibility_mi'].fillna(df['visibility_mi'].median())


df['severity_binary'] = df['severity'].apply(lambda x: 0 if x in [1, 2] else 1)



X = df.drop(columns=['severity', 'severity_binary'])
y = df['severity_binary']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1', verbose=1)
grid.fit(X_train_scaled, y_train)


best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("Best Parameters:", grid.best_params_)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


feature_names = X.columns

coefficients = best_model.coef_[0]


coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})


coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)


print("\nLogistic Regression Coefficients:")
print(coef_df)


Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Parameters: {'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'lbfgs'}

Accuracy: 0.5049

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.50      0.64     67452
           1       0.13      0.57      0.21      9060

    accuracy                           0.50     76512
   macro avg       0.51      0.53      0.43     76512
weighted avg       0.81      0.50      0.59     76512


Logistic Regression Coefficients:
         Feature  Coefficient
0  temperature_f     0.154118
1  visibility_mi    -0.064765


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

df = pd.read_csv("sc_accidents.csv")

df = df.drop(columns=['serial_number', 'end_lat', 'end_lng', 'start_lat', 'start_lng'])

df['temperature_f'] = df['temperature_f'].fillna(df['temperature_f'].median())
df['visibility_mi'] = df['visibility_mi'].fillna(df['visibility_mi'].median())

df['severity_binary'] = df['severity'].apply(lambda x: 0 if x in [1, 2] else 1)

X = df.drop(columns=['severity', 'severity_binary'])
y = df['severity_binary']

X = X.select_dtypes(include=['float64', 'int64'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1', verbose=1)
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("Best Parameters:", grid.best_params_)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

feature_names = X.columns
coefficients = best_model.coef_[0]
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefficients})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

print("\nLogistic Regression Coefficients:")
print(coef_df)

plt.figure(figsize=(10, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel("Coefficient Value")
plt.title("Logistic Regression Coefficients")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()




# DECISION TREES




# NAIVE BAYES

# K means CLUSTERING

# DEEP LEARNING

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv('/content/sc_accidents.csv')

features = ['temperature_f', 'visibility_mi']
target = 'severity'

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ann_model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(8, activation='relu'),
    Dense(1, activation='linear')
])

ann_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

ann_model.fit(X_train_scaled, y_train, epochs=20, batch_size=32, verbose=1)

y_pred_ann = ann_model.predict(X_test_scaled).round().astype(int).flatten()

print("ANN Accuracy:", accuracy_score(y_test, y_pred_ann))
print("Classification Report:\n", classification_report(y_test, y_pred_ann))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 0.3832 - mae: 0.3578
Epoch 2/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - loss: 0.1511 - mae: 0.2308
Epoch 3/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - loss: 0.1477 - mae: 0.2268
Epoch 4/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - loss: 0.1482 - mae: 0.2269
Epoch 5/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 43s 2ms/step - loss: 0.1486 - mae: 0.2279
Epoch 6/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 0.1488 - mae: 0.2299
Epoch 7/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - loss: 0.1484 - mae: 0.2280
Epoch 8/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 0.1485 - mae: 0.2288
Epoch 9/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - loss: 0.1480 - mae: 0.2281
Epoch 10/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 39s 2ms/step - loss: 0.1475 - mae: 0.2270
Epoch 11/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - loss: 0.1495 - mae: 0.2302
Epoch 12/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - loss: 0.1479 - m

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from tensorflow.keras.utils import to_categorical

# Convert y values to class indices starting from 0 if needed
y_train_cat = to_categorical(y_train - 1)
y_test_cat = to_categorical(y_test - 1)

ann_model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(8, activation='relu'),
    Dense(4, activation='softmax')  # 4 severity classes
])

ann_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

ann_model.fit(X_train_scaled, y_train_cat, epochs=20, batch_size=32, verbose=1)

# Predict classes
y_pred_proba = ann_model.predict(X_test_scaled)
y_pred_class = y_pred_proba.argmax(axis=1) + 1  # add 1 to match original labels

print("Classification Report:\n", classification_report(y_test, y_pred_class))


Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9361/9361 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - accuracy: 0.8511 - loss: 0.5337
Epoch 2/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.8649 - loss: 0.4667
Epoch 3/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.8657 - loss: 0.4655
Epoch 4/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.8646 - loss: 0.4678
Epoch 5/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8655 - loss: 0.4650
Epoch 6/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.8642 - loss: 0.4683
Epoch 7/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8650 - loss: 0.4666
Epoch 8/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8649 - loss: 0.4672
Epoch 9/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.8640 - loss: 0.4698
Epoch 10/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 0.8642 - loss: 0.4687
Epoch 11/20
9361/9361 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.8645 - loss: 0.4684
Epoch 12/20
9361/9361 ━━━━━━━━

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
-